In [8]:
!pip install qiskit qiskit-aer qiskit[visualization] matplotlib pylatexenc
try:
    import fqe
except ImportError:
    # TODO: Change to `pip install fqe --quiet` when FQE>0.1.0 is released.
    !pip install git+https://github.com/quantumlib/OpenFermion-FQE --quiet

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.4/131.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.8/670.8 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 67.7 MB/s eta 0:00:00


In [13]:
"""
Run IQPE on the Fermi-Hubbard Hamiltonian defined in the OpenFermion notebook.

The notebook uses:
    nsites = 4, U = 2.0, J = -1.0
    nele = 2, sz = 0  (two-particle, S_z=0 sector)

We convert the OpenFermion FermionOperator -> dense matrix -> SparsePauliOp
so it can be plugged straight into your existing iqpe() function.
"""

import numpy as np
import openfermion as of
from scipy.linalg import eigh
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import SparsePauliOp

from iqpe import iqpe
from utils import bitstring_to_eigenvalue


nsites = 4
U = 2.0
J = -1.0

hubbard_of = of.fermi_hubbard(1, nsites, tunneling=-J, coulomb=U, periodic=False)


nqubits = 2 * nsites                               # 8 qubits
H_dense = of.get_sparse_operator(hubbard_of, n_qubits=nqubits).toarray()
print(f"Hilbert-space dimension: {H_dense.shape}")


eigenvalues, eigenvectors = eigh(H_dense)
print("\nLowest 6 eigenvalues (reference):")
for i, e in enumerate(eigenvalues[:6]):
    print(f"  E[{i}] = {e:.6f}")


fh_pauli = SparsePauliOp.from_operator(H_dense).simplify()
print(f"\nNumber of Pauli terms: {len(fh_pauli)}")


k = 3
t = 2 * np.pi / 2**k
print(f"\nTime step: t = 2π / 2^{k} = {t:.6f}")


ground_state = list(eigenvectors[:, 0])

print("\n--- IQPE on ground state (num_bits=7) ---")
result_7 = iqpe(
    U=PauliEvolutionGate(fh_pauli, time=t),
    initial_state=ground_state,
    num_bits=7,
)
phase_7, bits_7 = result_7
eigenvalue_7 = bitstring_to_eigenvalue("".join(str(b) for b in bits_7), t, False)
print(f"  Phase   : {phase_7}")
print(f"  Bits    : {bits_7}")
print(f"  Estimated eigenvalue : {eigenvalue_7:.6f}")
print(f"  True ground energy   : {eigenvalues[0]:.6f}")

print("\n--- IQPE on ground state (num_bits=9) ---")
result_9 = iqpe(
    U=PauliEvolutionGate(fh_pauli, time=t),
    initial_state=ground_state,
    num_bits=9,
)
phase_9, bits_9 = result_9
eigenvalue_9 = bitstring_to_eigenvalue("".join(str(b) for b in bits_9), t, False)
print(f"  Phase   : {phase_9}")
print(f"  Bits    : {bits_9}")
print(f"  Estimated eigenvalue : {eigenvalue_9:.6f}")
print(f"  True ground energy   : {eigenvalues[0]:.6f}")


print("\n--- Sweep over eigenstates 0-3 (num_bits=7) ---")
for idx in range(4):
    init_state = list(eigenvectors[:, idx])
    res = iqpe(
        U=PauliEvolutionGate(fh_pauli, time=t),
        initial_state=init_state,
        num_bits=7,
    )
    phase, bits = res
    estimated_e = bitstring_to_eigenvalue("".join(str(b) for b in bits), t, False)
    print(
        f"  State {idx}: true E = {eigenvalues[idx]:8.4f} | "
        f"estimated E = {estimated_e:8.4f} | bits = {bits}"
    )

Hilbert-space dimension: (256, 256)

Lowest 6 eigenvalues (reference):
  E[0] = -3.069535
  E[1] = -3.069535
  E[2] = -2.875943
  E[3] = -2.819741
  E[4] = -2.236068
  E[5] = -2.236068

Number of Pauli terms: 25

Time step: t = 2π / 2^3 = 0.785398

--- IQPE on ground state (num_bits=7) ---
  Phase   : 0.375
  Bits    : [0, 1, 1, 0, 0, 0, 0]
  Estimated eigenvalue : -3.000000
  True ground energy   : -3.069535

--- IQPE on ground state (num_bits=9) ---
  Phase   : 0.375
  Bits    : [0, 1, 1, 0, 0, 0, 0, 0, 0]
  Estimated eigenvalue : -3.000000
  True ground energy   : -3.069535

--- Sweep over eigenstates 0-3 (num_bits=7) ---
  State 0: true E =  -3.0695 | estimated E =  -3.0000 | bits = [0, 1, 1, 0, 0, 0, 0]
  State 1: true E =  -3.0695 | estimated E =  -3.0000 | bits = [0, 1, 1, 0, 0, 0, 0]
  State 2: true E =  -2.8759 | estimated E =  -0.0000 | bits = [0, 0, 0, 0, 0, 0, 0]
  State 3: true E =  -2.8197 | estimated E =  -3.0000 | bits = [0, 1, 1, 0, 0, 0, 0]
